# 12. 짧은 영상 행동 분류

**실습 목표**  
짧은 동영상에서 프레임을 추출한 뒤 사전학습 VideoMAE로 행동 클래스를 예측한다.

**주요 Hugging Face 모델**  
`MCG-NJU/videomae-base-finetuned-kinetics`

> 이 노트북은 **파인튜닝 없이 사전학습 모델을 추론에 활용**하는 실습이다.  
> RTX 4060 8GB 환경을 고려했으며, CUDA가 없으면 CPU로 자동 전환하도록 구성하였다.

In [1]:
# uv add transformers accelerate av requests

In [2]:
import torch
print("PyTorch:", torch.__version__)

# 컴퓨터에 그래픽카드(GPU)가 있으면 계산이 훨씬 빨라져요. GPU가 있는지 확인!
print("CUDA available:", torch.cuda.is_available())

# transformers의 pipeline 기능은 GPU 번호를 숫자로 받아요 (0번 GPU, 없으면 -1 = CPU 사용)
DEVICE = 0 if torch.cuda.is_available() else -1
# torch 자체는 "cuda"(GPU) 또는 "cpu"라는 글자로 표시해요. 표기 방식만 다를 뿐 같은 의미예요
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", TORCH_DEVICE)

PyTorch: 2.14.0+cu126
CUDA available: True
device: cuda


In [3]:
import requests
# VIDEO_URL = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/sample_demo.mp4"
# 분류해볼 영상의 인터넷 주소예요
VIDEO_URL = "https://huggingface.co/spaces/LightwheelAI/README/resolve/main/assets/egosuite-open100k-promo.mp4"
r = requests.get(VIDEO_URL, timeout=60)  # 영상 데이터를 다운로드해요 (최대 60초 대기)
open("videos/sample.mp4", "wb").write(r.content)  # 다운로드한 영상을 컴퓨터 파일로 저장해요
print("saved videos/sample.mp4", len(r.content))

saved videos/sample.mp4 36545300


In [4]:
from transformers import pipeline

# 짧은 영상을 보고 "무슨 행동을 하고 있는지"(예: 달리기, 그림 그리기 등)를 맞추는 영상 분류 파이프라인이에요
video_classifier = pipeline(
    "video-classification",
    model="MCG-NJU/videomae-base-finetuned-kinetics",
    device=DEVICE
)

Loading weights:   0%|          | 0/162 [00:00<?, ?it/s]

[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                            | Status     | 
---------------------------------------------------------------+------------+-
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.query.bias | MISSING    | 
videomae.encoder.layer.{0...11}.attention.attention.value.bias | MISSING    | 
videomae.encoder.layer.{0...11}.attention.attention.key.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# uv pip install "torchcodec==0.14.0" \
#   --index-url https://download.pytorch.org/whl/cpu

# 이 명령어 꼭 해주기 

In [6]:
# 영상을 넣어서 가장 확률 높은 행동 5개(top_k=5)를 예측해봐요
try:
    result = video_classifier("videos/movie_sample.mp4", top_k=5)
    result
except Exception as e:
    
    print("샘플 URL 또는 비디오 디코더 환경에 따라 실행되지 않을 수 있다.")
    print("직접 mp4 파일을 업로드해 경로를 바꾸면 된다.")
    print(type(e).__name__, e)

In [7]:
# 예측 결과를 다시 한번 확인해요 (점수가 높을수록 그 행동일 가능성이 커요)
result

[{'score': 0.8864724636077881, 'label': 'drawing'},
 {'score': 0.002923408290371299, 'label': 'writing'},
 {'score': 0.001943436567671597, 'label': 'brush painting'},
 {'score': 0.0010970180155709386, 'label': 'archery'},
 {'score': 0.001060851151123643, 'label': 'contact juggling'}]

## 확장 과제
웹캠 영상을 3~5초 단위로 저장한 뒤 행동 분류 모델에 전달하는 미니 서비스를 설계해본다.